# Multiprocessing in Python

Multiprocessing allows you to create programs that can run concurrently (bypassing the Global Interpreter Lock or GIL) and use multiple CPU cores. This is essential for CPU-bound tasks.

## 1. Threading vs Multiprocessing

- **Threading**: Good for I/O-bound tasks (network, disk). Threads share the same memory space but are limited by the GIL (only one thread runs Python bytecode at a time).
- **Multiprocessing**: Good for CPU-bound tasks (heavy calculations). Creates separate memory spaces (processes) for each task, allowing true parallelism on multi-core CPUs.

## 2. Using `multiprocessing.Process`

The basic way to spawn a process.

In [ ]:
import multiprocessing
import time
import os

def worker_function(name):
    print(f"Worker {name} starting in process {os.getpid()}")
    time.sleep(2)
    print(f"Worker {name} finished")

if __name__ == '__main__':
    # Note: In Jupyter notebooks, multiprocessing can sometimes be tricky due to how cells are executed.
    # It's often better to define functions in a separate .py file and import them.
    # However, for simple examples, it might work or require 'fork' context on Unix.
    
    p1 = multiprocessing.Process(target=worker_function, args=('A',))
    p2 = multiprocessing.Process(target=worker_function, args=('B',))

    p1.start()
    p2.start()

    p1.join()
    p2.join()
    print("All processes finished")

## 3. Using `multiprocessing.Pool`

A Pool offers a convenient means of parallelizing the execution of a function across multiple input values, distributing the input data across processes (data parallelism).

In [ ]:
def square_number(x):
    return x * x

if __name__ == '__main__':
    with multiprocessing.Pool(processes=4) as pool:
        numbers = [1, 2, 3, 4, 5]
        results = pool.map(square_number, numbers)
        print(f"Squared numbers: {results}")

## 4. Inter-Process Communication (Queue)

Since processes don't share memory, we need special structures like Queues or Pipes to exchange data.

In [ ]:
def producer(queue):
    for i in range(5):
        queue.put(i)
        print(f"Produced {i}")
        time.sleep(0.5)

def consumer(queue):
    while True:
        item = queue.get()
        if item is None: # Sentinel value to stop
            break
        print(f"Consumed {item}")

if __name__ == '__main__':
    q = multiprocessing.Queue()
    p1 = multiprocessing.Process(target=producer, args=(q,))
    p2 = multiprocessing.Process(target=consumer, args=(q,))

    p1.start()
    p2.start()

    p1.join()
    q.put(None) # Signal consumer to stop
    p2.join()